In [ ]:
## importing packages
from enum import auto
from statistics import median
import statistics
from unicodedata import decimal
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from pprint import pprint
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
from hyperopt import hp, fmin, tpe, space_eval, Trials
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import os
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
## Open the file
Perovskites = pd.read_csv(r"C:\Users\Perovskites - machine learning\database_perovskitas_ML_multitarget_hotencoding_importantvariabNOVO.txt", 
                                delimiter = "\t", decimal = ",")

## remove columns based on correlation plot, 'perovskite' não dá pra classificar só em dois tipos
##  e 'am/ca' (ver caderno o pq)
Perovskites = Perovskites.drop(["Total_amount"], axis=1)

## remove rows outliers and stuff
Perovskites = Perovskites.drop([37,52,154, 56, 15, 65], axis=0, inplace=False)

#### columns removed based on shap feature importance
Perovskites = Perovskites.drop(['S_II','S_II_amount_mL'], axis=1)

### usando one hot encoding em algumas categóricas
Perovskites_encoded = pd.get_dummies(Perovskites, columns=['A_source','B1_source', 'B2_source', 'X_source'])
print(Perovskites_encoded)

#trocando features para logfeatures - features com range muito alto
Perovskites_encoded['Diameter_nm'] = np.log10(Perovskites_encoded["Diameter_nm"])
Perovskites_encoded['Time_min'] = np.log10(Perovskites_encoded["Time_min"])
Perovskites_encoded['B2_amount_mmol'] = np.log10(Perovskites_encoded["B2_amount_mmol"])
Perovskites_encoded['Temperature_K'] = np.log10(Perovskites_encoded["Temperature_K"])
Perovskites_encoded['Bandgap'] = np.log10(Perovskites_encoded["Bandgap"])
Perovskites_encoded['Tolerance_factor_new'] = np.log10(Perovskites_encoded["Tolerance_factor_new"])

## after remove columns from shap
x = Perovskites_encoded.drop(['B1_amount_mmol', 'X_amount_mmol', 'Time_min', 'Bandgap', 'Amine_amount_mmol'], axis=1)
y = Perovskites_encoded[[ 'B1_amount_mmol',  'X_amount_mmol', 'Time_min','Bandgap', 'Amine_amount_mmol']]

np.random.seed(100)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=25) # 75% training and 25% test

In [ ]:
'''
#hyperopt
####### to optimize hyperparameters, do the lines below
# Define the search space
space = {
    'max_depth': hp.choice('max_depth', range(1, 500)),
    'max_features': hp.choice('max_features', [1.0, 'sqrt', 'auto']),
    'max_leaf_nodes': hp.choice('max_leaf_nodes', range(2, 500)),
    'min_samples_leaf': hp.choice('min_samples_leaf', range(2, 10)),
    'min_samples_split': hp.choice('min_samples_split', range(2, 10)),
    'min_weight_fraction_leaf': hp.choice('min_weight_fraction_leaf', [0.1])
}

cv = KFold(n_splits=5, random_state=25, shuffle=True)
# Define the objective function
def objective(params):
    dt = DecisionTreeRegressor(**params)
    scores = cross_val_score(dt, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-scores.mean())  # Negative mean squared error to minimize
    return rmse

# Run the hyperparameter optimization
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)

# Get the best hyperparameters
best_params = space_eval(space, best)

# Create a Random Forest regressor with the best hyperparameters
dt = DecisionTreeRegressor(**best_params)
scores = cross_val_score(dt, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')

dt.fit(X_train, y_train)

# Prediction on test set
y_pred = dt.predict(X_test)
y_pred_train = dt.predict(X_train)

print(best_params)
'''

In [ ]:
# Create the parameter grid based on the results of random search - using hyperopt
best_params = {
              'max_depth': 480, 
              'max_features':'auto', 
              'max_leaf_nodes': 280,
              'min_weight_fraction_leaf': 0.1, 
              'min_samples_leaf': 5,
              'min_samples_split': 7 
}
# Create a Random Forest regressor with the best hyperparameters
dt = DecisionTreeRegressor(**best_params)
dt.fit(X_train, y_train)

# Prediction on test set
y_pred = dt.predict(X_test)
y_pred_train = dt.predict(X_train)

In [ ]:
'''
#gridsearch
####### to optimize hyperparameters, do the lines below

param_grid = {
                   'max_depth': range(2,500,50),
                   'max_features': ('auto', 'sqrt'),
                   'max_leaf_nodes': range(2,500,50),
                   'min_samples_leaf': range(2,10),
                   'min_weight_fraction_leaf': [0.1],
                   'min_samples_split': range(2,10)
}


# Create a based model
dt = DecisionTreeRegressor()
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = dt, param_grid = param_grid, 
                          cv = 5, n_jobs = -1, verbose = 2)

# Fit the grid search to the data
grid_search.fit(X_train, y_train)
pprint(grid_search.best_params_)

# prediction on test set
y_pred=grid_search.best_estimator_.predict(X_test)

## training metrics
y_pred_train=grid_search.best_estimator_.predict(X_train)

print(best_params)


### after defining best hyperparameters
param_grid = {
              'max_depth': [485], 
              'max_features':['sqrt'], 
              'max_leaf_nodes': [320],
              'min_weight_fraction_leaf': [0.1], 
              'min_samples_leaf': [4],
              'min_samples_split': [9] 
}

# Create a based model
dt = DecisionTreeRegressor()
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = dt, param_grid = param_grid, 
                          cv = 5, n_jobs = -1, verbose = 2)

# Fit the grid search to the data
grid_search.fit(X_train, y_train)
pprint(grid_search.best_params_)

# prediction on test set
y_pred=grid_search.best_estimator_.predict(X_test)

## training metrics
y_pred_train=grid_search.best_estimator_.predict(X_train)
'''


In [ ]:
### using raw_values to obtain individual metrics values, if i want the sum of all target erros, just del multioutput='raw_values'
### train metrics
MAE_train = pd.DataFrame(metrics.mean_absolute_error(y_train, y_pred_train, multioutput='raw_values'))
RMSE_train = pd.DataFrame(np.sqrt(metrics.mean_squared_error(y_train, y_pred_train, multioutput='raw_values')))
R2_train = pd.DataFrame(r2_score(y_train, y_pred_train, multioutput='raw_values'))

train_metrics_dt = pd.concat([MAE_train, RMSE_train, R2_train], axis='columns')
train_metrics_dt.columns = ['MAE_train','RMSE_train','R2_train']
print(train_metrics_dt)

### test metrics
MAE_test= pd.DataFrame(metrics.mean_absolute_error(y_test, y_pred, multioutput='raw_values'))
RMSE_test = pd.DataFrame(np.sqrt(metrics.mean_squared_error(y_test, y_pred, multioutput='raw_values')))
R2_test =pd.DataFrame(r2_score(y_test, y_pred, multioutput='raw_values'))

test_metrics_dt = pd.concat([MAE_test, RMSE_test, R2_test], axis='columns')
test_metrics_dt.columns = ['MAE_test', 'RMSE_test','R2_test']
print(test_metrics_dt)